[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Daniel-534/IntroduccionAstronomiaPractica/tree/main/CirculosPrincipales-CoordenadasCelestes/Analisis.ipynb)

In [ ]:
"""
Sun Ephemeris Query — JPL Horizons via astroquery
==================================================
Target  : Sol (Sun) [ID=10]
Observer: 6.2675°E, 75.2675°S, 1495 m  (lon/lat/elev)
Period  : 2026-04-03 00:00 UT → 2026-04-04 00:00 UT
Step    : 1 minute  (1441 epochs)

Columns retrieved
-----------------
  datetime_str   — epoch label from Horizons
  datetime_jd    — Julian Date
  RA             — Astrometric Right Ascension  [deg, J2000/ICRF]
  DEC            — Astrometric Declination       [deg, J2000/ICRF]
  AZ             — Apparent Azimuth              [deg, N→E convention]
  EL             — Apparent Elevation            [deg]

Quantities used (Horizons codes)
---------------------------------
  1  → Astrometric RA & DEC (J2000/ICRF)
  4  → Apparent AZ & EL (airless, i.e. no refraction correction)

Install requirements
--------------------
  pip install astroquery pandas
"""

import pandas as pd
from astroquery.jplhorizons import Horizons

# ── Observer location ────────────────────────────────────────────────────────
# Format expected by Horizons: {'lon': deg_E, 'lat': deg_N, 'elevation': km}
# Note: negative latitude → Southern Hemisphere
LOCATION = {
    "lon":       6.267452683656837,   # degrees East
    "lat":      -75.267452683656837,  # degrees North (negative = South)
    "elevation": 1.495,               # km above WGS-84 ellipsoid
}

# ── Epochs ───────────────────────────────────────────────────────────────────
EPOCHS = {
    "start": "2026-04-03 00:00",
    "stop":  "2026-04-04 00:00",
    "step":  "1m",               # 1-minute cadence
}

# ── Query ────────────────────────────────────────────────────────────────────
print("Querying JPL Horizons …")
obj = Horizons(
    id="10",           # Sun
    id_type="id",
    location=LOCATION,
    epochs=EPOCHS,
)

eph = obj.ephemerides(
    quantities="1,4",        # 1 = Astrometric RA/DEC, 4 = Apparent AZ/EL
    skip_daylight=False,     # include nighttime rows (Sun below horizon)
    apparent="airless",      # geometric AZ/EL, no atmospheric refraction
    airmass_lessthan=99,     # no airmass cut-off
    extra_precision=False,
)

# ── Build DataFrame ──────────────────────────────────────────────────────────
# Convert AstroPy table → pandas, then keep only the columns we care about
df_raw = eph.to_pandas()

df = df_raw[["datetime_str", "datetime_jd", "RA", "DEC", "AZ", "EL"]].copy()

df.rename(columns={
    "datetime_str": "Epoch (UT)",
    "datetime_jd":  "JD",
    "RA":           "Astrometric RA [deg]",
    "DEC":          "Astrometric DEC [deg]",
    "AZ":           "Apparent AZ [deg]",
    "EL":           "Apparent EL [deg]",
}, inplace=True)

# ── Inspect & save ───────────────────────────────────────────────────────────
print(f"\nShape  : {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\nFirst 5 rows:\n{df.head().to_string(index=False)}")
print(f"\nLast 5 rows:\n{df.tail().to_string(index=False)}")

output_path = "sun_ephemeris_2026-04-03.csv"
df.to_csv(output_path, index=False)
print(f"\nSaved → {output_path}")